# UniFormer-S 3D + trọng số Kinetics — phân loại u gan trên MRI đa pha

Huấn luyện **UniFormer-S 3D** phân loại **7 lớp** tổn thương gan trên volume **8 pha** MRI,
khởi tạo từ trọng số học trên **video Kinetics-400**.

Trục lát của volume đóng vai trò trục thời gian của mô hình video, còn 8 pha MRI vào làm
**kênh đầu vào**. Lớp `patch_embed1` vì thế không nhận được trọng số cũ (8 kênh ≠ 3 kênh RGB)
và được train từ đầu; toàn bộ giá trị chuyển giao nằm ở stage 2–4.

Cấu hình: `configs/uniformer_s.yaml`. Không sửa gì trong notebook — mọi siêu tham số đọc từ
file đó.

## Cần mount gì

| | |
|---|---|
| **Cache lưới `128×128×16`** | Sinh bởi `configs/preprocess_cghnet.yaml`; mô hình nhận `112×112×14` sau khi cắt |
| **Internet: BẬT** | để tải `uniformer_small_k400_16x8.pth` (~200 MB) từ HuggingFace |

Tải xong nên lưu trọng số thành Kaggle Dataset và mount cho các session sau, đỡ tải lại.

## Năm cổng chạy TRƯỚC khi cam kết fold nào

Mỗi cổng chặn một chế độ hỏng **im lặng** — loại lỗi vẫn chạy trơn và vẫn ra số trông hợp lý.

| cổng | chặn gì |
|---|---|
| **A** trọng số | nạp trọng số **thất bại một phần** mà không có ngoại lệ nào |
| **B** hình học | mô hình chạy ở kích thước đầu vào sai |
| **C** ngân sách | phát hiện chi phí quá cao **sau khi** đã cam kết cả session |
| **D** sampler | khoá lấy mẫu lại không thật sự có tác dụng |
| **E** augment | augmentation phá cấu trúc đa pha của dữ liệu |

Chạy tuần tự từ trên xuống. Cổng nào đỏ thì **dừng**, đừng chạy mục 2.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1]                    # ⚠️ 1 fold KHÔNG kết luận được gì (CI ~±0.19). Nó chỉ
                               # dùng để LOẠI, và chỉ khi thấp hẳn. Xem bar ở mục 3.
CONFIG_NAME = "uniformer_s.yaml"
PREPROCESS_NAME = "preprocess_cghnet.yaml"
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
PRE = load_yaml(REPO / "configs" / PREPROCESS_NAME)
M = CFG["model"]

INNER = tuple(PRE["target_size"])                       # [X, Y, Z] = 112, 112, 14
MARGIN = tuple(PRE["crop_margin_voxels"])
GRID = tuple(s + 2 * m for s, m in zip(INNER, MARGIN))  # 128, 128, 16
assert tuple(CFG["data"]["crop_size"]) == INNER, "data.crop_size lệch target_size của cache"

# Thứ tự của MODEL là (D, H, W) với D = trục lát; cache là [X, Y, Z]. Đổi một lần ở đây và
# dùng biến này ở mọi cổng, để không ai phải nhẩm lại giữa chừng.
MODEL_SIZE = (INNER[2], INNER[0], INNER[1])             # (14, 112, 112)

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"model     : uniformer {M['variant']} · patch_embed1_stride {M['patch_embed1_stride']}")
print(f"            drop_path {M['drop_path_rate']} · drop_rate {M['drop_rate']} · "
      f"head_dropout {M['head_dropout']}")
print(f"loss      : {CFG['loss']['name']} gamma {CFG['loss']['gamma']} · "
      f"class_weights {CFG['loss']['class_weights']} · smoothing {CFG['loss']['label_smoothing']}")
print(f"sampler   : {CFG['data']['sampling']}")
print(f"augment   : edge {CFG['data']['augment']['edge_prob']} · "
      f"emboss {CFG['data']['augment']['emboss_prob']} · "
      f"filter {CFG['data']['augment']['filter_prob']}")
print(f"hình học  : cache {GRID} -> model nhận {INNER} [X,Y,Z] = {MODEL_SIZE} (D,H,W)")

## 1. Cache

Cần cache có lưới **`128×128×16`** (`configs/preprocess_cghnet.yaml`): mô hình nhận
`112×112×14`, phần dư mỗi phía là lề cho phép cắt ngẫu nhiên lúc train và cắt giữa lúc suy luận.

⚠️ Cache hình học khác **không dùng được** — `data.crop_size` phải khớp `target_size` của
chính cache đang mount, và cell dưới `assert` điều đó trước khi chạy tiếp.

In [ ]:
import json as _json

import numpy as np

CAN = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": list(INNER),
    "crop_margin_voxels": list(MARGIN),
}

ung_vien, CACHE_DIR = [], None
for meta_path in sorted(Path("/kaggle/input").rglob("cache_meta.json")):
    try:
        meta = _json.loads(meta_path.read_text("utf-8"))
    except Exception:  # noqa: BLE001 - chỉ để liệt kê chẩn đoán
        continue
    khop = all(meta.get(k) == v for k, v in CAN.items())
    ung_vien.append((meta_path.parent, meta, khop))
    if khop and CACHE_DIR is None:
        CACHE_DIR = meta_path.parent

print(f"=== {len(ung_vien)} cache tìm thấy dưới /kaggle/input ===")
for path, meta, khop in ung_vien:
    print(f"  {'✓ khớp  ' if khop else '  --    '}  {path}")
    print(f"          size={meta.get('target_size')} lề={meta.get('crop_margin_voxels')} "
          f"align={meta.get('align_phases')} crop={meta.get('crop_mode')}")

if CACHE_DIR is None:
    raise RuntimeError(
        "Chưa mount cache đúng hình học.\n"
        f"  Cần cache có {CAN}\n"
        "  Chạy notebooks/18_build_cache_cghnet.ipynb trước (CPU, Accelerator = None,\n"
        "  ~20 phút), lưu output thành Dataset, rồi mount vào đây.\n"
        "  Cache lưới khác KHÔNG dùng được — bảng ở trên liệt kê mọi cache đang mount."
    )

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498"
with np.load(next(CACHE_DIR.glob("*.npz"))) as z:
    shape = tuple(z["image"].shape)
assert shape == (8, *GRID), f"mảng {shape}, cần {(8, *GRID)} — cache này sai hình học"
print(f"\ncache ✓ · {CACHE_DIR} · {n_npz} ca · mảng {shape}")

## 1b. Trọng số Kinetics

`uniformer_small_k400_16x8.pth` — UniFormer-S huấn luyện trên video **Kinetics-400**, lấy từ
HuggingFace `Sense-X/uniformer_video`.

Cell dưới ưu tiên bản đã mount làm Dataset (không tốn băng thông, chạy được cả khi tắt
Internet); không có thì tải, và cần **bật Internet**.

In [ ]:
import urllib.request

from src.models.uniformer3d import HF_BASE_URL, PRETRAINED_FILENAMES

FILENAME = PRETRAINED_FILENAMES[M["variant"]]

# 1) Bản đã mount làm Dataset — không tốn băng thông, và dùng được cả khi tắt Internet.
CKPT = next(iter(sorted(Path("/kaggle/input").rglob(FILENAME))), None)

# 2) Không có thì tải.
if CKPT is None:
    CKPT = Path("/kaggle/working/pretrained") / FILENAME
    CKPT.parent.mkdir(parents=True, exist_ok=True)
    if not CKPT.exists():
        url = HF_BASE_URL + FILENAME
        print("tải:", url)
        # Tải vào `.part` rồi mới đổi tên. Tải dở dang mà để nguyên tên đích thì lần chạy
        # sau `CKPT.exists()` là True và nó dùng luôn file cụt — cổng A sẽ nổ với thông báo
        # về kiến trúc, hoàn toàn sai hướng.
        part = CKPT.with_suffix(CKPT.suffix + ".part")
        try:
            urllib.request.urlretrieve(url, part)
            assert part.stat().st_size > 50e6, (
                f"file tải về chỉ {part.stat().st_size / 1e6:.1f} MB, quá nhỏ — "
                "gần như chắc chắn là trang lỗi HTML chứ không phải checkpoint"
            )
            part.replace(CKPT)
        except Exception as exc:  # noqa: BLE001 - cần thông báo dạy được, không phải traceback
            part.unlink(missing_ok=True)
            raise RuntimeError(
                f"Không tải được {FILENAME}. Hai nguyên nhân thường gặp:\n"
                "  1. Internet đang TẮT — bật ở panel bên phải (Settings > Internet).\n"
                "  2. HuggingFace chặn tạm — tải tay rồi upload thành Kaggle Dataset,\n"
                f"     cell này tự tìm theo tên file {FILENAME} dưới /kaggle/input.\n"
                f"  URL: {url}"
            ) from exc

os.environ["LLDMMRI_PRETRAINED_PATH"] = str(CKPT)
print(f"trọng số ✓ · {CKPT} · {CKPT.stat().st_size / 1e6:.0f} MB")

## Cổng A ⚠️⚠️ — trọng số có thật sự vào model không

**`load_state_dict(strict=False)` không báo lỗi khi không khoá nào khớp.** Model vẫn dựng
được, vẫn train, vẫn ra số — chỉ là "có pretrained" lặng lẽ thành "khởi tạo ngẫu nhiên", đúng
thứ mà cả cấu hình này dựa vào.

⚠️ Cổng này in **khoá NÀO** thiếu, **không** in tỉ lệ phần trăm. Một kiến trúc lệch nhẹ so với
file trọng số vẫn có thể khớp ~85% khoá — dư sức qua bất kỳ ngưỡng phần trăm nào, trong khi
những khoá bị bỏ lại đúng là phần quan trọng.

Chỉ hai tiền tố được phép thiếu, và cả hai vì hình học không khớp được:

* `patch_embed1.` — 8 pha MRI ≠ 3 kênh RGB
* `head.` — 7 lớp ≠ 400 lớp Kinetics

In [ ]:
import torch

from src.models.uniformer3d import DROPPED_PREFIXES, build_uniformer3d, load_kinetics_weights

tham_so = {k: v for k, v in M.items() if k not in ("name", "require_pretrained")}
tham_so["pretrained_path"] = None
model = build_uniformer3d(**tham_so, require_pretrained=False)

n_tham_so = sum(p.numel() for p in model.parameters())
bao_cao = load_kinetics_weights(model, CKPT)   # NỔ nếu thiếu khoá ngoài hai tiền tố trên

print(f"tham số model : {n_tham_so / 1e6:.2f}M")
print(f"khoá nạp được : {len(bao_cao['loaded'])}")
print(f"khoá thiếu    : {len(bao_cao['missing'])}  (chỉ được phép {DROPPED_PREFIXES})")
for k in bao_cao["missing"]:
    print("   thiếu :", k)
print(f"khoá dư trong checkpoint: {len(bao_cao['unexpected'])}")
for k in bao_cao["unexpected"][:10]:
    print("   dư    :", k)

assert bao_cao["loaded"], "⛔ KHÔNG nạp được khoá nào — đây là run from scratch trá hình"
assert all(k.startswith(DROPPED_PREFIXES) for k in bao_cao["missing"])
# Một mạng UniFormer-S có ~300 khoá; nạp được dưới 100 nghĩa là khớp nhầm nhánh nào đó.
assert len(bao_cao["loaded"]) > 100, f"chỉ nạp {len(bao_cao['loaded'])} khoá — quá ít"
print("\ncổng A ✓ — trọng số Kinetics ĐÃ vào model")

## Cổng B ⚠️⚠️⚠️ — hình dạng thật qua từng stage

Một mạng chạy ở kích thước đầu vào sai **không nổ và không cảnh báo** — nó vẫn hội tụ và vẫn
ra số trông hợp lý, chỉ là thấp hơn đáng lẽ. Cell này bắt hook vào từng `patch_embed` để đọc
shape **thật**, rồi đối chiếu với `stage_token_counts` tính bằng tay.

⚠️ Đây cũng là chỗ đọc ra nút thắt chi phí. `patch_embed1` có stride `(1,2,2)` nên **không hạ
mẫu trục lát**: stage 3 nhận **2744** token, so với **1568** ở cấu hình gốc của file trọng số
(`16×224×224`). Stage 3–4 dùng attention **toàn cục**, nên số token vào đó quyết định giờ chạy.

In [ ]:
from src.models.uniformer3d import stage_token_counts

thuc_te = []
hooks = [
    getattr(model, f"patch_embed{i}").register_forward_hook(
        lambda _m, _i, out: thuc_te.append(tuple(out.shape[2:]))
    )
    for i in range(1, 5)
]
# Chạy trên GPU: stage 3 có ~2744 token attention toàn cục, forward trên CPU tốn vài
# phút. Đưa model về CPU ngay sau đó để cổng C tự quản việc chuyển thiết bị.
_dev_b = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(_dev_b).eval()
with torch.no_grad():
    ra = model(torch.zeros(1, 8, *INNER, device=_dev_b))
for h in hooks:
    h.remove()
model.cpu()
ra = ra.cpu()

tinh_tay = stage_token_counts(MODEL_SIZE, M["patch_embed1_stride"])
pretrained = stage_token_counts((16, 224, 224), (2, 4, 4))

print(f"{'stage':<6}{'thực tế (D,H,W)':>20}{'tính tay':>18}{'token':>10}{'pretrained':>12}")
for i, (that, tay, pre) in enumerate(zip(thuc_te, tinh_tay, pretrained), start=1):
    loai = "CBlock" if i <= 2 else "SABlock"
    print(f"{i} {loai:<5}{str(that):>18}{str(tay):>18}"
          f"{that[0] * that[1] * that[2]:>10}{pre[0] * pre[1] * pre[2]:>12}")

assert thuc_te == tinh_tay, f"⛔ shape thật {thuc_te} lệch tính tay {tinh_tay}"
assert tuple(ra.shape) == (1, CFG["model"]["num_classes"])

n3 = tinh_tay[2][0] * tinh_tay[2][1] * tinh_tay[2][2]
n3_pre = pretrained[2][0] * pretrained[2][1] * pretrained[2][2]
print(f"\nstage 3 (attention TOÀN CỤC, depth 8): {n3} token so với {n3_pre} của bản pretrained"
      f"  ⇒ {n3 / n3_pre:.2f}× token, ~{(n3 / n3_pre) ** 2:.1f}× chi phí attention")
print(f"đầu ra: {tuple(ra.shape)}")
print("\ncổng B ✓")

## Cổng C ⚠️ — ngân sách, ĐO THẬT

Phải **đo** s/epoch, không được suy từ số GFLOPs: chi phí thật phụ thuộc attention toàn cục,
băng thông bộ nhớ và tốc độ nạp dữ liệu, và ước lượng từ FLOPs lệch nhiều lần theo cả hai chiều.
Đo trước thì mất 2 phút; đoán sai thì mất cả một session.

Quá **60 s/epoch** ⇒ ngân sách 5 fold không lọt. Khoá thoát: đổi `model.patch_embed1_stride`
sang `[2, 2, 2]` (hạ lát 14→7, stage 3 còn 1372 token, ~¼ chi phí attention) rồi chạy lại từ
cổng A.

In [ ]:
import time

from src.train.run import build_loaders

train_loader, val_loader, train_labels = build_loaders(CFG, FOLDS[0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).train()
opt = torch.optim.AdamW(model.parameters(), lr=float(CFG["train"]["lr"]))
scaler = torch.amp.GradScaler("cuda", enabled=bool(CFG["train"]["amp"]))

N_DO = 8
t0 = None
for i, batch in enumerate(train_loader):
    if i == 2:          # bỏ 2 batch đầu: nạp worker + cudnn benchmark
        torch.cuda.synchronize() if device.type == "cuda" else None
        t0 = time.perf_counter()
    if i == 2 + N_DO:
        break
    x = batch["image"].to(device, non_blocking=True)
    y = batch["label"].to(device, non_blocking=True)
    with torch.amp.autocast("cuda", enabled=bool(CFG["train"]["amp"])):
        loss = torch.nn.functional.cross_entropy(model(x), y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
if device.type == "cuda":
    torch.cuda.synchronize()

assert t0 is not None, (
    f"⛔ chỉ có {len(train_loader)} batch, cần > 2 để bỏ warm-up. Kiểm lại batch_size."
)
giay_moi_batch = (time.perf_counter() - t0) / N_DO
n_batch = len(train_loader)
s_epoch = giay_moi_batch * n_batch * 1.15      # +15% cho vòng val, ước từ các run trước
gio_fold = s_epoch * int(CFG["train"]["epochs"]) / 3600

print(f"thiết bị      : {device}")
print(f"batch train   : {n_batch} · batch_size {CFG['data']['batch_size']}")
print(f"giây/batch    : {giay_moi_batch:.3f}")
print(f"giây/epoch    : {s_epoch:.1f}   (đã cộng ~15% cho val)")
print(f"giờ/fold      : {gio_fold:.2f}  ({CFG['train']['epochs']} epoch)")
print(f"giờ/{len(FOLDS)} fold  : {gio_fold * len(FOLDS):.2f}")
print(f"giờ/5 fold    : {gio_fold * 5:.2f}")
print("\nràng buộc Kaggle: session tối đa 12h · quota GPU ~30h/tuần")
print("⚠️ Số này ĐO THẬT trên GPU đang dùng — đừng suy ra từ GFLOPs hay từ cấu hình khác.")

# ⚠️ Đây là `print`, KHÔNG phải `assert` — cổng C không chặn notebook, nó chỉ nói ngân sách
# thật để bạn quyết định. Cấu hình hiện tại là lựa chọn có ý thức, không phải sai sót.
print()
print(f"1 fold {gio_fold:.1f}h — {'lọt' if gio_fold < 12 else 'KHÔNG lọt'} một session 12h")
print(f"5 fold {gio_fold * 5:.1f}h — "
      f"{'lọt' if gio_fold * 5 <= 30 else 'VƯỢT'} quota 30h/tuần"
      + ("" if gio_fold * 5 <= 30 else " ⇒ phải trải qua nhiều tuần"))

if s_epoch > 60:
    print("\ncổng C ✓ (có ghi chú) — chi phí cao, và đó là hệ quả đã biết của việc giữ")
    print("   `patch_embed1_stride` không hạ mẫu trục lát. Đây là bài toán KẾ HOẠCH, không")
    print("   phải lỗi: chia số fold sang nhiều tuần quota.")
    print("   Nếu CHỦ ĐỘNG muốn đánh đổi thì `[2, 2, 2]` hạ lát 14→7 và rẻ hơn ~2–3×, nhưng")
    print("   nó đổi kiến trúc nên mọi con số trước đó không so trực tiếp được nữa.")
else:
    print("\ncổng C ✓")

model.cpu()
del opt, scaler
torch.cuda.empty_cache() if device.type == "cuda" else None

## Cổng D ⚠️ — `data.sampling` có thật sự đổi phân bố nhãn không

`WeightedRandomSampler` nhận trọng số rồi lặng lẽ bỏ qua nếu bị truyền sai chỗ, và `DataLoader`
với `shuffle=True` sẽ chạy y như cũ mà không có cảnh báo nào. Cell này **đếm nhãn thực tế được
rút** qua một epoch và so với thành phần lớp thật.

⚠️ **Cấu hình này bật HAI lớp cân bằng cùng lúc:** trọng số lớp trong loss
(`loss.class_weights: effective_number`) **và** lấy mẫu lại (`data.sampling: sqrt`). Chúng cộng
dồn, nên sau khi có kết quả hãy kiểm cán cân dự đoán:

```
python -m src.eval.weak_classes --run-dir runs/uniformer_s
```

Nếu lớp hiếm bị dự đoán **thừa** (tỉ lệ đoán/thật vượt ~1.4×) thì đã đẩy quá tay, và
`data.sampling: instance` là ablation **một khoá** để tách hai lớp đó ra.

In [ ]:
import collections

from src.data.taxonomy import SHORT_NAMES
from src.train.run import SAMPLING_EXPONENTS

MODE = str(CFG["data"]["sampling"])
sampler = train_loader.sampler
print("sampler:", type(sampler).__name__, "· data.sampling =", MODE)

if MODE == "instance":
    # `instance` là ablation HỢP LỆ, nên BỎ QUA chứ không làm nổ notebook.
    print("\ncổng D — BỎ QUA: `instance` không lấy mẫu lại, không có gì để kiểm")
else:
    assert sampler is not None and hasattr(sampler, "weights"), "⛔ sampler chưa được nối vào"

    lab = np.asarray(train_labels)
    cnt = np.bincount(lab, minlength=len(SHORT_NAMES)).astype(float)
    w = np.asarray(sampler.weights, dtype=float)
    assert len(w) == len(lab), f"⛔ {len(w)} trọng số nhưng {len(lab)} mẫu train"
    co = cnt > 0
    q = SAMPLING_EXPONENTS[MODE]

    # --- D1: kiểm TRỌNG SỐ, tất định --------------------------------------------
    # Đây là chỗ duy nhất kiểm được chắc chắn. Số ca *được rút* dao động ±5–6 ca giữa các
    # epoch (lấy mẫu có hoàn lại), nên mọi kiểm định dựa trên con số tuyệt đối đó đều có
    # tỉ lệ báo động sai đáng kể ngay cả khi sampler hoàn toàn đúng.
    w_lop = np.full(len(cnt), np.nan)
    for c in np.flatnonzero(co):
        wc = w[lab == c]
        assert np.allclose(wc, wc[0]), f"⛔ lớp {SHORT_NAMES[c]} có trọng số không đều nhau"
        w_lop[c] = wc[0]
    ky = cnt[co] ** (-q)
    assert np.allclose(w_lop[co] / w_lop[co][0], ky / ky[0]), (
        f"⛔ tỉ lệ trọng số giữa các lớp KHÔNG khớp count^(-{q}) — sai số mũ hoặc sai mode"
    )
    print(f"  D1 ✓ trọng số mỗi lớp ∝ count^(-{q}) — khớp chính xác, không phụ thuộc may rủi")

    # --- D2: đếm thật qua một epoch, để đọc bằng mắt -----------------------------
    N = len(lab)
    s = cnt[co] ** (1.0 - q)
    kv = np.zeros(len(cnt))
    kv[co] = N * s / s.sum()          # kỳ vọng số ca được rút mỗi epoch
    lay = collections.Counter(lab[list(iter(sampler))])
    print(f"\n  {'lớp':<9}{'thật':>7}{'kỳ vọng':>10}{'được lấy':>11}{'tỉ lệ':>8}")
    for c in sorted(SHORT_NAMES):
        print(f"  {SHORT_NAMES[c]:<9}{int(cnt[c]):>7}{kv[c]:>10.1f}{lay[c]:>11}"
              f"{lay[c] / max(cnt[c], 1):>8.2f}")

    # --- D3: kiểm định HƯỚNG, chịu được nhiễu ------------------------------------
    # So TỈ LỆ giữa lớp hiếm nhất và lớp đông nhất, không so số tuyệt đối với chính nó.
    # Hai tỉ lệ này lệch nhau theo hệ thống (kỳ vọng ~1.9 lần) nên phép so rất bền.
    dong = int(cnt.argmax())
    hiem = int(np.where(co, cnt, np.inf).argmin())
    ti_d, ti_h = lay[dong] / cnt[dong], lay[hiem] / cnt[hiem]
    assert ti_h > ti_d, (
        f"⛔ tỉ lệ lấy của lớp hiếm nhất ({SHORT_NAMES[hiem]} {ti_h:.2f}) không cao hơn lớp "
        f"đông nhất ({SHORT_NAMES[dong]} {ti_d:.2f}) — phân bố không dịch về phía cân bằng"
    )

    if lay[hiem] <= cnt[hiem]:
        p = kv[hiem] / N
        print(f"\n  ⚠ {SHORT_NAMES[hiem]} được lấy {lay[hiem]} ca, không vượt {int(cnt[hiem])} ca"
              f" thật — nhưng kỳ vọng là {kv[hiem]:.0f} ± {np.sqrt(N * p * (1 - p)):.0f} ca,"
              " nên đây là nhiễu lấy mẫu, KHÔNG phải lỗi.")

    print(f"\ncổng D ✓ — `{MODE}` dịch phân bố về phía cân bằng: tỉ lệ lấy của "
          f"{SHORT_NAMES[hiem]} là {ti_h:.2f} so với {ti_d:.2f} của {SHORT_NAMES[dong]}")

## Cổng E ⚠️ — augmentation có phá cấu trúc đa pha không

Chẩn đoán u gan trên MRI đa pha dựa vào cường độ **tương đối giữa các pha** — ngấm rồi thải,
ngấm tiến triển, viền ngấm. Nên một augmentation vẽ **tham số ngẫu nhiên riêng cho từng pha**
sẽ đổ nhiễu thẳng lên chính tín hiệu phân biệt, mà vẫn cho ảnh trông bình thường.

Bất biến phải giữ: **cùng một tham số ngẫu nhiên cho cả 8 pha.**

⚠️ **Bất biến đó KHÔNG phải "8 pha ra kết quả giống nhau".** Hai chuyện khác nhau:

| phép | trục pha | hai pha giống nhau ⇒ đầu ra giống nhau? |
|---|---|---|
| `edge` · `emboss` · `sharpen` | kernel `(1,3,3,1)`, **không** chạm trục pha | **có** |
| `blur` · `unsharp` | `filter_spatial_only: false` ⇒ σ broadcast ra **cả trục pha** | **không** |

`blur`/`unsharp` làm mờ **cả trục pha** là hành vi mặc định của config, nên chúng *phải* cho
pha 0 khác pha 3 kể cả khi đầu vào hai pha bằng nhau — hai pha đó có láng giềng khác nhau.
Kỳ vọng: `filter_prob × (blur_prob + unsharp_prob)` = 0.40 × 0.30 = **12%** số lượt.

Cell này vì thế kiểm ở **hai tầng**:

* **E1a** — bật `filter_spatial_only=True` để không phép nào trộn pha. Khi đó hai pha giống
  nhau **bắt buộc** cho đầu ra giống nhau. Đây mới là phép kiểm "cùng tham số", và nó là
  `assert`.
* **E1b** — chế độ mặc định: **đếm** số lượt trộn pha và đối chiếu kỳ vọng 12%, rồi kiểm trực
  tiếp rằng ba phép *không* trộn pha thì thật sự không trộn.

**E2** kiểm một chế độ hỏng khác: phép xoay lấp giá trị 0 vào góc sẽ làm mẫu train mang dải
đệm ở rìa mà mẫu val không có — một lệch phân bố train/val có hệ thống ở mọi bước huấn luyện.
Lệch tỉ lệ voxel 0 quá 0.02 là dấu hiệu `rotate_mode` đặt sai.

In [ ]:
import numpy as np

from src.data.transforms import (
    RandomAppearance,
    build_train_transform,
    pil_kernel_filter,
    sobel_magnitude,
)
from src.train.run import build_loaders  # lặp lại import để cell này tự đứng được
from src.utils.seed import set_seed

chain = build_train_transform(CFG["data"]["augment"], CFG["data"]["crop_size"])
app = [t for t in chain.transforms if isinstance(t, RandomAppearance)]
assert len(app) == 1, f"⛔ có {len(app)} RandomAppearance trong chuỗi, cần đúng 1"
assert isinstance(chain.transforms[-1], RandomAppearance), "⛔ phải nằm CUỐI, sau RandomCrop3D"
print("chuỗi transform:", [type(t).__name__ for t in chain.transforms])

AUG = CFG["data"]["augment"]
P_TRON = AUG["filter_prob"] * (AUG["blur_prob"] + AUG["unsharp_prob"])
N_LUOT = 200


def _do_lech_pha(phep) -> tuple[int, int]:
    """Đếm (số lượt bị áp phép, số lượt pha 0 khác pha 3) khi pha 3 sao chép pha 0."""
    set_seed(1337)
    n_ap = lech = 0
    for _ in range(N_LUOT):
        goc = torch.randn(8, *GRID)
        goc[3] = goc[0]
        ra = phep({"image": goc.clone()})["image"]
        if not torch.allclose(ra, goc):
            n_ap += 1
            if not torch.allclose(ra[0], ra[3], atol=1e-4):
                lech += 1
    return n_ap, lech


# --- E1a: BẤT BIẾN THẬT — cùng tham số cho 8 pha ------------------------------
# Kiểm ở chế độ spatial_only: không phép nào trộn trục pha, nên hai pha giống nhau BẮT BUỘC
# cho đầu ra giống nhau. Lệch ở đây = có tham số ngẫu nhiên vẽ riêng cho từng pha.
sach = RandomAppearance(
    edge_prob=AUG["edge_prob"],
    emboss_prob=AUG["emboss_prob"],
    filter_prob=AUG["filter_prob"],
    blur_prob=AUG["blur_prob"],
    sharpen_prob=AUG["sharpen_prob"],
    unsharp_prob=AUG["unsharp_prob"],
    filter_spatial_only=True,
)
n_ap, lech = _do_lech_pha(sach)
print(f"\nE1a · filter_spatial_only=True (không phép nào trộn pha)")
print(f"  {N_LUOT} lượt: {n_ap} lượt bị áp phép "
      f"({100 * n_ap / N_LUOT:.0f}%, kỳ vọng ~40%) · {lech} lượt lệch pha")
assert lech == 0, (
    f"⛔ {lech} lượt áp KHÁC NHAU giữa các pha dù đã tắt trộn pha. Có tham số ngẫu nhiên vẽ "
    "riêng cho từng pha — augmentation đang phá cấu trúc đa pha. DỪNG."
)
print("  ✓ cùng tham số cho cả 8 pha")

# --- E1b: chế độ trung thực — trộn pha là CHỦ Ý, chỉ ĐẾM chứ không assert -----
n_ap2, tron = _do_lech_pha(app[0])
ky_vong = P_TRON * N_LUOT
print(f"\nE1b · filter_spatial_only={app[0].filter_spatial_only} (mặc định của config)")
print(f"  {N_LUOT} lượt: {n_ap2} lượt bị áp phép · {tron} lượt pha 0 ≠ pha 3")
print(f"  kỳ vọng trộn pha = filter_prob × (blur + unsharp) = "
      f"{AUG['filter_prob']} × {AUG['blur_prob'] + AUG['unsharp_prob']:.2f} = "
      f"{100 * P_TRON:.0f}% ⇒ ~{ky_vong:.0f} lượt")
if abs(tron - ky_vong) > 3 * (ky_vong**0.5):
    print(f"  ⚠ {tron} lệch xa {ky_vong:.0f} quá 3·√n — đọc lại cây quyết định của "
          "`RandomAppearance`, có thể sai xác suất nhánh")
else:
    print("  ✓ khớp kỳ vọng — blur/unsharp trộn trục pha, đúng như config đặt")

# Và kiểm TRỰC TIẾP rằng ba phép không-trộn-pha thật sự không trộn.
arr = np.random.default_rng(0).standard_normal((8, *GRID)).astype(np.float32)
arr[3] = arr[0]
for ten, phep in (
    ("edge", sobel_magnitude),
    ("emboss", lambda a: pil_kernel_filter(a, "emboss")),
    ("sharpen", lambda a: pil_kernel_filter(a, "sharpen")),
):
    ra = phep(arr)
    assert np.allclose(ra[0], ra[3], atol=1e-4), f"⛔ {ten} áp khác nhau giữa các pha"
    print(f"  {ten:<8} không trộn pha ✓")

# --- E2: rìa train so với val -----------------------------------------------
# ⚠️ Đo với BA AUGMENT LỌC TẮT. Lý do: `emboss` có kernel tổng bằng 0 nên vùng phẳng ra
# **đúng 0**, và rìa khối cắt bám tổn thương thì thường phẳng — đủ để cổng này báo động sai
# ~10% số lượt. Thứ cần đo ở đây là voxel 0 do **hình học** (xoay/tịnh tiến lấp 0), nên phải
# tách hẳn hai nguồn ra.
import copy

# Chỉ dựng THÊM MỘT loader (train, đã tắt ba khoá lọc). Phía val dùng lại `val_loader`
# của cổng C: `build_val_transform` chỉ cắt giữa, không augment gì, nên nó đã đúng là thứ
# cần so. Mỗi loader giữ 4 worker; dựng dư là ăn RAM của Kaggle vô ích.
cfg_hh = copy.deepcopy(CFG)
for _k in ("edge_prob", "emboss_prob", "filter_prob"):
    cfg_hh["data"]["augment"][_k] = 0
train_hh, _val_bo, _ = build_loaders(cfg_hh, FOLDS[0])
val_hh = val_loader


def ti_le_0_o_ria(img):
    """img: [B, 8, X, Y, Z] — tỉ lệ voxel ~0 trong dải 4 voxel sát rìa mặt phẳng."""
    ria = torch.cat([img[:, :, :4].flatten(), img[:, :, -4:].flatten(),
                     img[:, :, :, :4].flatten(), img[:, :, :, -4:].flatten()])
    return float((ria.abs() < 1e-6).float().mean())


def do_ria(loader, n_batch=6):
    vals = []
    for i, b in enumerate(loader):
        if i >= n_batch:
            break
        vals.append(ti_le_0_o_ria(b["image"]))
    return float(np.mean(vals))


tr, va = do_ria(train_hh), do_ria(val_hh)
print(f"\nE2 · voxel 0 ở rìa (ba augment lọc TẮT, chỉ còn hình học)")
print(f"  train {tr:.4f} · val {va:.4f} · lệch {abs(tr - va):.4f}")
assert abs(tr - va) < 0.02, (
    "⛔ lệch phân bố rìa train/val — kiểm `rotate_mode` có đúng `nearest` không. "
    "Xoay lấp 0 ở góc làm mẫu train mang dải đệm ở rìa mà mẫu val không có."
)
print("  ✓ không có dải đệm 0 lệch giữa train và val")
print("\ncổng E ✓")

## 2. Train

Năm cổng phải xanh hết trước khi chạy cell này. `resume: true` nên ngắt session giữa chừng
thì chạy lại là đọc tiếp từ `last.pt`.

In [ ]:
from src.train.run import TRAIN_RESULT_KEYS, train

# Giải phóng model của cổng A–C và MỌI loader của các cổng. Mỗi loader giữ 4 worker
# process; để chúng sống suốt lúc train là ăn RAM và tranh CPU với worker thật.
for _ten in ("model", "train_loader", "val_loader", "train_hh", "_val_bo", "sach", "app", "chain"):
    globals().pop(_ten, None)
import gc

gc.collect()
torch.cuda.empty_cache()

print("khoá train() trả về:", TRAIN_RESULT_KEYS)
results = {}
for fold in FOLDS:
    print(f"\n{'=' * 70}\nfold {fold}\n{'=' * 70}")
    results[fold] = train(CFG_PATH, fold)
    # ⚠️ `best_macro_f1`, KHÔNG phải `macro_f1`. Đọc sai tên thì `KeyError` nổ ở đúng dòng này,
    # tức SAU KHI đã train xong cả fold, và vòng lặp dừng luôn trước fold kế tiếp.
    print("fold %d xong: macro-F1 %.4f @ epoch %d"
          % (fold, results[fold]["best_macro_f1"], results[fold]["best_epoch"]))

## 3. Kết quả từng fold, và bar quyết định đã chốt trước

⚠️ **`/kaggle/working` bị xoá khi session kết thúc.** Chạy mục này ở session khác với session
train thì thư mục run không còn — cell **sẽ nổ kèm hướng dẫn** thay vì in một bảng rỗng. Một
cell im lặng trông y như một cell chưa có gì để in, nên nó tệ hơn một cell nổ.

In [ ]:
import csv as _csv
import re as _re


def _tim_run():
    trong_session = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
    if list(trong_session.glob("fold*/metrics_best.json")):
        return trong_session
    for cand in sorted(Path("/kaggle/input").glob("*/**/fold*/metrics_best.json")):
        print("dùng run đã mount:", cand.parent.parent)
        return cand.parent.parent
    raise RuntimeError(
        "Không thấy fold nào có metrics_best.json. Đã tìm ở "
        + str(trong_session)
        + " và /kaggle/input/**/fold*/. `/kaggle/working` bị xoá khi session kết thúc, nên"
        " nếu bạn train ở session TRƯỚC thì phải upload thư mục run thành Kaggle Dataset rồi"
        " mount vào đây. Hoặc chạy lại cell train ở mục 2 — `resume: true` nên nó đọc tiếp"
        " từ last.pt."
    )


OUT = _tim_run()
rows = []
for d in sorted(OUT.glob("fold*")):
    mp = d / "metrics_best.json"
    if not mp.exists():
        continue
    m = _json.loads(mp.read_text("utf-8"))
    # ⚠️ Tên thư mục có HAI dạng: `fold1_<digest>` do train() sinh, và `fold_1` sau khi gói
    # lại mang về. `split("_")[0]` cho "fold" ở dạng thứ hai => int("") NỔ. Dùng regex.
    _m_fold = _re.search(r"fold_?(\d+)", d.name)
    assert _m_fold, f"không đọc được số fold từ tên thư mục {d.name!r}"
    fold = int(_m_fold.group(1))
    day = 0
    log = d / "train_log.csv"
    if log.exists():
        vals = [float(r["val_loss"]) for r in _csv.DictReader(log.open(encoding="utf-8"))]
        day = int(np.argmin(vals)) + 1 if vals else 0
    # ⚠️ `metrics_best.json` ghi **cohen_kappa**, không phải `kappa`. Đọc sai tên thì
    # `.get` trả nan và cột kappa in ra nan **im lặng**, không có cảnh báo nào.
    assert "cohen_kappa" in m, f"{mp} không có khoá cohen_kappa, chỉ có {sorted(m)}"
    rows.append(
        (fold, m["macro_f1"], m["cohen_kappa"], m.get("epoch", 0), day, m.get("per_class_f1"))
    )

assert rows, "⛔ không đọc được fold nào"

print(f"{'fold':<6}{'macro-F1':>10}{'kappa':>9}{'epoch':>7}{'val_loss đáy':>14}")
for fold, f1, kap, ep, day, _ in sorted(rows):
    print(f"{fold:<6}{f1:>10.4f}{kap:>9.4f}{ep:>7}{day:>14}")

if len(rows) > 1:
    print(f"\ntrung bình {len(rows)} fold: {float(np.mean([r[1] for r in rows])):.4f}")

# F1 từng lớp — thứ đáng đọc nhất, vì macro-F1 là trung bình của cả 7 lớp nên một lớp yếu
# kéo nó xuống nhiều hơn mức trực giác gợi ý.
from src.data.taxonomy import SHORT_NAMES

print(f"\n{'lớp':<9}" + "".join(f"{'fold ' + str(r[0]):>10}" for r in sorted(rows)))
for c in sorted(SHORT_NAMES):
    hang = "".join(
        f"{(r[5][c] if r[5] else float('nan')):>10.3f}" for r in sorted(rows)
    )
    print(f"{SHORT_NAMES[c]:<9}{hang}")

print(f"\n⚠️ ĐANG CÓ {len(rows)} FOLD.")
if len(rows) == 1:
    print("   Một fold có n≈80 ca, khoảng tin cậy 95% rộng khoảng ±0.19. Con số ở đây KHÔNG")
    print("   kết luận được gì, kể cả khi nó cao: phương sai giữa các fold của bài toán này")
    print("   lớn hơn phần lớn hiệu ứng cần đo. Chạy thêm fold trước khi tin.")
elif len(rows) < 5:
    print(f"   {len(rows)} fold đủ để LOẠI một cấu hình, chưa đủ để CHỌN nó. Một kết quả dương")
    print("   ở cỡ mẫu này chỉ có nghĩa 'chưa loại được'.")
else:
    print("   Đủ 5 fold. Con số báo cáo được là bản GỘP out-of-fold, không phải trung bình các")
    print("   fold — trung bình các fold không có khoảng tin cậy đúng nghĩa vì mỗi fold là một")
    print("   tập nhỏ khác nhau.")

print("\nHai chẩn đoán nên xem tiếp, cả hai chạy trên CPU từ xác suất đã lưu:")
print("   · tỉ lệ lỗi có biên hẹp — nói ngưỡng/hiệu chỉnh có cứu được gì không")
print("   · top-2 của các lớp yếu — nói biểu diễn có mã hoá được lớp đó không")
print("\n     python -m src.eval.run          --run-dir runs/uniformer_s")
print("     python -m src.eval.weak_classes --run-dir runs/uniformer_s")

## 4. Gói mang về

In [ ]:
import shutil

goi = Path("/kaggle/working") / f"{EXPERIMENT}_results"
if goi.exists():
    shutil.rmtree(goi)
goi.mkdir(parents=True)

for d in sorted(OUT.glob("fold*")):
    dest = goi / d.name
    dest.mkdir()
    # `train()` ghi **config_used.json**, không phải `config.yaml` — pattern cũ khớp 0 file
    # nên config lặng lẽ không được gói, và run mang về mất mất dấu vết cấu hình.
    for pattern in ("metrics_best.json", "train_log.csv", "val_probs_*.npz", "config_used.json"):
        for f in d.glob(pattern):
            shutil.copy2(f, dest / f.name)

shutil.make_archive(str(goi), "zip", goi)
print("đã gói:", goi.with_suffix(".zip"))
print("⚠️ KHÔNG gói best.pt/last.pt — checkpoint quá lớn để mang theo. Cần chúng cho suy luận")
print("   về sau thì tải riêng từ Output của session này.")
for p in sorted(goi.rglob("*")):
    print("  ", p.relative_to(goi))